# RAG Evaluation with Ragas & DeepEval

A hands-on notebook comparing **OpenAI** and **Anthropic** answer quality on the SQuAD v2 dataset using two modern LLM evaluation frameworks.

**Dataset**: [SQuAD v2](https://rajpurkar.github.io/SQuAD-explorer/) — reading-comprehension QA with context passages and ground-truth answers. Each sample has exactly what a RAG pipeline produces: a retrieved context chunk, a generated answer, and a reference answer.

**Models compared**
- Generator: `gpt-4o-mini` vs `claude-haiku-4-5` (fast + cheap, good for benchmarking)
- Judge LLM: `gpt-4o-mini` (same for both, so scoring is fair)

**Install**: `pip install -r requirements.txt`

## Package Capabilities Overview

Both packages go far beyond simple groundedness/faithfulness checks.

### Ragas — RAG-specific metrics

| Metric | What it measures | Needs ground truth? |
|--------|-----------------|---------------------|
| **Faithfulness** | Every claim in the answer is supported by the retrieved context | No |
| **ContextPrecision** | Retrieved chunks are ranked with the most relevant ones first | Yes |
| **LLMContextRecall** | The retrieved context contains all info needed to answer correctly | Yes |
| **FactualCorrectness** | Facts in the answer match the reference (precision + recall of claims) | Yes |
| **AnswerRelevancy** | The answer directly addresses the question (not off-topic) | No |
| **NoiseSensitivity** | Answer stability when irrelevant chunks are added to context | Yes |
| **SemanticSimilarity** | Embedding similarity between answer and reference | Yes |
| **AspectCritic** | Pass/fail on a natural-language aspect (e.g., "Is the answer polite?") | No |
| **SimpleCriteriaScore** | 0–1 score on a custom natural-language criterion | No |
| Agent metrics | ToolCallAccuracy, AgentGoalAccuracy for agentic pipelines | Varies |

### DeepEval — General-purpose LLM testing

| Metric | What it measures | Needs ground truth? |
|--------|-----------------|---------------------|
| **FaithfulnessMetric** | Answer is grounded in the retrieved context | No |
| **AnswerRelevancyMetric** | Answer addresses the user's question | No |
| **ContextualRelevancyMetric** | Each retrieved chunk is relevant to the query | No |
| **ContextualPrecisionMetric** | Retrieved chunks are ranked with relevant ones first | Yes |
| **ContextualRecallMetric** | Retrieved context covers the expected answer | Yes |
| **HallucinationMetric** | Answer contains fabricated facts not in the context | No |
| **GEval** | Flexible rubric-based evaluation using any criteria you define | Varies |
| **SummarizationMetric** | Quality of document summaries | No |
| **BiasMetric** | Detects gender, racial, or political bias | No |
| **ToxicityMetric** | Detects harmful or offensive content | No |
| Agent metrics | ToolCorrectnessMetric, TaskCompletionMetric for agents | Varies |

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv

# Must be set BEFORE importing deepeval, or deepeval prompts you to log in
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

for _root in [Path.cwd(), *Path.cwd().parents]:
    _env = _root / ".env"
    if _env.is_file():
        load_dotenv(_env)
        break

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip()
assert OPENAI_API_KEY, "OPENAI_API_KEY missing from .env"
assert ANTHROPIC_API_KEY, "ANTHROPIC_API_KEY missing from .env"

OPENAI_GEN_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
ANTHROPIC_GEN_MODEL = "claude-haiku-4-5-20251001"
JUDGE_MODEL = os.environ.get("OPENAI_JUDGE_MODEL", "gpt-4o-mini")
N_SAMPLES = 15  # small enough to be cheap, big enough to be meaningful

print(f"OpenAI generator:    {OPENAI_GEN_MODEL}")
print(f"Anthropic generator: {ANTHROPIC_GEN_MODEL}")
print(f"Judge model:         {JUDGE_MODEL}")
print(f"N samples:           {N_SAMPLES}")

### Initialize API Clients

> **Key Ragas gotcha**: Unlike many other tools, Ragas does **not** auto-read `OPENAI_API_KEY` from the environment. You must explicitly wrap your LLM using `LangchainLLMWrapper`. Ragas also needs an embeddings client for metrics like `AnswerRelevancy` that compare answer and question semantically.

In [ ]:
import openai
from openai import OpenAI
from anthropic import Anthropic
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import OpenAIEmbeddings

# Direct clients — used to generate answers
oai_client = OpenAI(api_key=OPENAI_API_KEY)
ant_client = Anthropic(api_key=ANTHROPIC_API_KEY)

# Ragas LLM + embeddings wrapper — used internally by Ragas metrics
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model=JUDGE_MODEL, api_key=OPENAI_API_KEY))
ragas_embeddings = OpenAIEmbeddings(client=openai.OpenAI(api_key=OPENAI_API_KEY))

print("All clients initialized.")

## Dataset — SQuAD v2

SQuAD v2 is ideal for RAG evaluation because each row contains:
- `context` — the retrieved passage (our "RAG context")
- `question` — the user query
- `answers.text` — ground-truth answer(s)

We filter to **answerable** rows (SQuAD v2 includes unanswerable questions too) and take a small shuffle-stable subset.

In [ ]:
from datasets import load_dataset

ds = load_dataset("rajpurkar/squad_v2")
val_ds = ds["validation"]

answerable_val = val_ds.filter(lambda row: bool(row["answers"]["text"]))
subset = answerable_val.shuffle(seed=42).select(range(N_SAMPLES))

print(f"Total answerable validation rows: {len(answerable_val)}")
print(f"Using subset of: {len(subset)} rows")
print("\nSample row:")
subset[0]

## Generate Answers — OpenAI & Anthropic

We use a strict system prompt so the models answer from context only, simulating a RAG generator. This makes faithfulness metrics meaningful — any answer not grounded in the context is a genuine failure.

In [ ]:
QA_SYSTEM = (
    "Answer the reading-comprehension question using only the provided passage. "
    "Reply with the shortest correct answer phrase. No preamble, no quotes."
)


def generate_openai(context: str, question: str) -> str:
    resp = oai_client.chat.completions.create(
        model=OPENAI_GEN_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": QA_SYSTEM},
            {"role": "user", "content": f"Passage:\n{context}\n\nQuestion:\n{question}"},
        ],
    )
    return resp.choices[0].message.content.strip()


def generate_anthropic(context: str, question: str) -> str:
    resp = ant_client.messages.create(
        model=ANTHROPIC_GEN_MODEL,
        system=QA_SYSTEM,
        max_tokens=256,
        messages=[{"role": "user", "content": f"Passage:\n{context}\n\nQuestion:\n{question}"}],
    )
    return resp.content[0].text.strip()

In [ ]:
rows = []
print(f"Generating answers for {N_SAMPLES} questions...")
for i, row in enumerate(subset):
    ground_truth = row["answers"]["text"][0]
    oai_ans = generate_openai(row["context"], row["question"])
    time.sleep(0.1)
    ant_ans = generate_anthropic(row["context"], row["question"])
    rows.append({
        "question":         row["question"],
        "context":          row["context"],
        "ground_truth":     ground_truth,
        "openai_answer":    oai_ans,
        "anthropic_answer": ant_ans,
    })
    if (i + 1) % 5 == 0 or i == 0:
        print(f"  [{i + 1}/{N_SAMPLES}] Q: {row['question'][:60]}")
        print(f"          GT:  {ground_truth}")
        print(f"          OAI: {oai_ans}")
        print(f"          ANT: {ant_ans}")

print(f"\nDone — {len(rows)} rows generated.")

In [ ]:
pd.set_option("display.max_colwidth", 80)
pd.DataFrame([
    {
        "Question":     r["question"][:70],
        "Ground Truth": r["ground_truth"],
        "OpenAI":       r["openai_answer"],
        "Anthropic":    r["anthropic_answer"],
    }
    for r in rows[:5]
])

## Ragas Evaluation

Ragas wraps evaluation in an `EvaluationDataset` of `SingleTurnSample` objects. Each sample must contain:

| Field | Maps to (SQuAD) |
|-------|-----------------|
| `user_input` | `question` |
| `retrieved_contexts` | `[context]` (single chunk list) |
| `response` | generated answer |
| `reference` | `answers.text[0]` (single string, not a list) |

**Metrics we'll use** — all 5 require the LLM wrapper we set up earlier:

| Metric | Requires embeddings? |
|--------|---------------------|
| `Faithfulness` | No |
| `ContextPrecision` | No |
| `LLMContextRecall` | No |
| `FactualCorrectness` | No |
| `AnswerRelevancy` | **Yes** — generates paraphrase questions and checks cosine similarity |

In [ ]:
from ragas import EvaluationDataset, evaluate as ragas_evaluate
from ragas.metrics.collections import (
    Faithfulness,
    ContextPrecision,
    LLMContextRecall,
    FactualCorrectness,
    AnswerRelevancy,
)

ragas_metrics = [
    Faithfulness(llm=ragas_llm),
    ContextPrecision(llm=ragas_llm),
    LLMContextRecall(llm=ragas_llm),
    FactualCorrectness(llm=ragas_llm),
    AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings),
]


def build_ragas_dataset(rows: list[dict], answer_key: str) -> EvaluationDataset:
    return EvaluationDataset.from_list([
        {
            "user_input":         r["question"],
            "retrieved_contexts": [r["context"]],   # list of context chunks
            "response":           r[answer_key],
            "reference":          r["ground_truth"],  # single string, NOT a list
        }
        for r in rows
    ])


print("Ragas metrics ready:", [type(m).__name__ for m in ragas_metrics])

In [ ]:
print("Running Ragas on OpenAI answers...")
oai_ragas_ds = build_ragas_dataset(rows, "openai_answer")
oai_ragas_result = ragas_evaluate(
    oai_ragas_ds, metrics=ragas_metrics, llm=ragas_llm, embeddings=ragas_embeddings
)

print("\nRunning Ragas on Anthropic answers...")
ant_ragas_ds = build_ragas_dataset(rows, "anthropic_answer")
ant_ragas_result = ragas_evaluate(
    ant_ragas_ds, metrics=ragas_metrics, llm=ragas_llm, embeddings=ragas_embeddings
)

print("\nDone!")

In [ ]:
oai_ragas_df = oai_ragas_result.to_pandas()
ant_ragas_df = ant_ragas_result.to_pandas()

# Identify numeric metric columns (exclude sample-level text fields)
_skip = {"user_input", "retrieved_contexts", "response", "reference", "id"}
ragas_metric_cols = [c for c in oai_ragas_df.columns if c not in _skip]

ragas_comparison = pd.DataFrame({
    "OpenAI":    {col: round(float(oai_ragas_df[col].mean()), 3) for col in ragas_metric_cols},
    "Anthropic": {col: round(float(ant_ragas_df[col].mean()), 3) for col in ragas_metric_cols},
})
ragas_comparison["Winner"] = ragas_comparison.idxmax(axis=1)
ragas_comparison.index.name = "Metric"
print("Ragas comparison (mean across", N_SAMPLES, "samples)")
ragas_comparison

## DeepEval Evaluation

DeepEval uses `LLMTestCase` objects. Key fields:

| Field | Purpose | Maps to (SQuAD) |
|-------|---------|------------------|
| `input` | User's question | `question` |
| `actual_output` | LLM's generated answer | generated answer |
| `expected_output` | Ground truth (for precision/recall) | `answers.text[0]` |
| `retrieval_context` | Chunks that were retrieved | `[context]` |
| `context` | Ideal knowledge (for HallucinationMetric only) | `[context]` |

> **`context` vs `retrieval_context`**: `HallucinationMetric` uses `context` (the golden knowledge base). All other RAG metrics (`Faithfulness`, `ContextualPrecision`, etc.) use `retrieval_context`. In our case they're the same SQuAD passage.

**Metrics we'll use**:
- `FaithfulnessMetric` — same concept as Ragas Faithfulness, good for cross-framework comparison
- `AnswerRelevancyMetric` — is the answer on-topic?
- `ContextualPrecisionMetric` — are the retrieved chunks relevant?
- `ContextualRecallMetric` — does context cover the expected answer?
- `HallucinationMetric` — does the answer fabricate facts?
- `GEval(Conciseness)` — custom rubric: is the answer concise?

In [ ]:
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    HallucinationMetric,
    GEval,
)
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

_threshold = 0.5
_model = JUDGE_MODEL

deepeval_metrics = [
    FaithfulnessMetric(model=_model, threshold=_threshold),
    AnswerRelevancyMetric(model=_model, threshold=_threshold),
    ContextualPrecisionMetric(model=_model, threshold=_threshold),
    ContextualRecallMetric(model=_model, threshold=_threshold),
    HallucinationMetric(model=_model, threshold=_threshold),
    GEval(
        name="Conciseness",
        criteria=(
            "Evaluate whether the answer is concise. "
            "A concise answer gives the necessary information without verbosity, repetition, or filler phrases."
        ),
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=_model,
        threshold=_threshold,
    ),
]

print("DeepEval metrics:", [type(m).__name__ if not isinstance(m, GEval) else f"GEval({m.name})" for m in deepeval_metrics])

In [ ]:
def build_deepeval_cases(rows: list[dict], answer_key: str) -> list[LLMTestCase]:
    return [
        LLMTestCase(
            input=r["question"],
            actual_output=r[answer_key],
            expected_output=r["ground_truth"],
            retrieval_context=[r["context"]],  # for Faithfulness, Contextual* metrics
            context=[r["context"]],            # for HallucinationMetric
        )
        for r in rows
    ]


def run_deepeval(test_cases: list[LLMTestCase], metrics: list) -> dict[str, float]:
    """Run each metric on each test case; return per-metric mean score."""
    scores: dict[str, list[float]] = {}
    for m in metrics:
        name = m.name if isinstance(m, GEval) else type(m).__name__
        scores[name] = []

    for i, tc in enumerate(test_cases):
        for m in metrics:
            name = m.name if isinstance(m, GEval) else type(m).__name__
            try:
                m.measure(tc)
                scores[name].append(m.score)
            except Exception as e:
                print(f"  Warning [{name}] case {i}: {e}")
        if (i + 1) % 5 == 0:
            print(f"  [{i + 1}/{len(test_cases)}] done")

    return {k: round(float(np.mean(v)), 3) if v else 0.0 for k, v in scores.items()}

In [ ]:
print("Running DeepEval on OpenAI answers...")
oai_deepeval_cases = build_deepeval_cases(rows, "openai_answer")
oai_deepeval_scores = run_deepeval(oai_deepeval_cases, deepeval_metrics)

print("\nRunning DeepEval on Anthropic answers...")
ant_deepeval_cases = build_deepeval_cases(rows, "anthropic_answer")
ant_deepeval_scores = run_deepeval(ant_deepeval_cases, deepeval_metrics)

print("\nDone!")

In [ ]:
deepeval_comparison = pd.DataFrame({
    "OpenAI":    oai_deepeval_scores,
    "Anthropic": ant_deepeval_scores,
})
deepeval_comparison["Winner"] = deepeval_comparison.idxmax(axis=1)
deepeval_comparison.index.name = "Metric"
print("DeepEval comparison (mean across", N_SAMPLES, "samples)")
deepeval_comparison

## Final Comparison — Side by Side

Combining both frameworks lets us see whether they agree: if Ragas and DeepEval both say Anthropic is more faithful, that's a stronger signal than one framework alone.

In [ ]:
# Shared metric names that appear in both frameworks (faithfulness + answer relevancy)
ragas_comp = ragas_comparison[["OpenAI", "Anthropic"]].copy()
ragas_comp.index = [f"Ragas/{m}" for m in ragas_comp.index]

deepeval_comp = deepeval_comparison[["OpenAI", "Anthropic"]].copy()
deepeval_comp.index = [f"DeepEval/{m}" for m in deepeval_comp.index]

all_metrics = pd.concat([ragas_comp, deepeval_comp])
all_metrics["OpenAI wins"] = all_metrics["OpenAI"] > all_metrics["Anthropic"]
all_metrics["Diff (OAI - ANT)"] = (all_metrics["OpenAI"] - all_metrics["Anthropic"]).round(3)

print(f"OpenAI wins:    {all_metrics['OpenAI wins'].sum()} / {len(all_metrics)}")
print(f"Anthropic wins: {(~all_metrics['OpenAI wins']).sum()} / {len(all_metrics)}")
all_metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
_blue, _orange = "#4472C4", "#ED7D31"

for ax, (df, title) in zip(axes, [
    (ragas_comparison[["OpenAI", "Anthropic"]], "Ragas"),
    (deepeval_comparison[["OpenAI", "Anthropic"]], "DeepEval"),
]):
    x = np.arange(len(df))
    w = 0.35
    ax.bar(x - w / 2, df["OpenAI"],    w, label="OpenAI",    color=_blue,   alpha=0.85)
    ax.bar(x + w / 2, df["Anthropic"], w, label="Anthropic", color=_orange, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(df.index, rotation=30, ha="right", fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score (0–1)")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    # Add score labels on bars
    for bar in ax.patches:
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2, h + 0.02,
            f"{h:.2f}", ha="center", va="bottom", fontsize=7
        )

plt.suptitle(
    f"OpenAI ({OPENAI_GEN_MODEL}) vs Anthropic ({ANTHROPIC_GEN_MODEL})\n"
    f"Judge: {JUDGE_MODEL}  |  Dataset: SQuAD v2  |  N={N_SAMPLES}",
    y=1.03, fontsize=12
)
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to model_comparison.png")

## Key Takeaways

### When to use Ragas
- **RAG pipeline–specific**: Ragas was built specifically for RAG evaluation. Use it when you want metrics that explicitly reason about the retrieval step (`ContextPrecision`, `LLMContextRecall`).
- **FactualCorrectness**: Ragas decomposes answers into atomic claims and checks each against the reference — more fine-grained than simple string matching.
- **Requires LangChain wrapper**: Unlike most tools, Ragas does not auto-pick up `OPENAI_API_KEY`. You must wrap the LLM with `LangchainLLMWrapper`.

### When to use DeepEval
- **Broader coverage**: DeepEval covers safety (bias, toxicity), summarization, agents, and custom rubrics (`GEval`) in a single framework.
- **`HallucinationMetric`**: Unlike Ragas Faithfulness (claims must be entailed by context), DeepEval's Hallucination detects contradictions — useful when you want to flag outputs that actively state something false.
- **`GEval`**: Define any evaluation criterion in plain English — e.g., conciseness, politeness, domain-specific compliance.

### Cross-framework agreement
- When both Ragas and DeepEval agree on a winner for a metric (e.g., both say Anthropic is more faithful), that's a strong signal.
- Disagreements usually come from different internal prompts/rubrics for the same concept — comparing them side-by-side is a good sanity check.

### Practical tips
- **Cost**: Each metric calls an LLM. With 5 Ragas + 6 DeepEval metrics on 15 samples, expect ~150–200 LLM calls for evaluation alone.
- **Sampling**: 15–30 samples is enough for exploratory work; 100+ for production decisions.
- **Judge bias**: Using the same judge model for both generators keeps comparisons fair, but check if the judge has a known preference for its own outputs.